In [1]:
import pandas as pd
import glob
import numpy as np


In [2]:
# ### Add missing Cargo ID when it is not present in the CSV file

# mapping = {
#     "DIPUTADOS NACIONALES": 3,
#     "DIPUTADOS PROVINCIALES": 6,
#     "CONCEJALES": 10,
#     "SENADORES PROVINCIALES": 5,
#     "SENADORES NACIONALES": 2,
#     "INTENDENTES": 7,
#     "DIPUTADO NACIONAL": 3,
#     "SENADOR NACIONAL": 2,
#     "DIPUTADO PROVINCIAL": 6,
#     "SENADOR PROVINCIAL": 5
# }

# filepaths = [    "./../datos/Solo_ResultadosProvisorios_bkp/ResultadosElectorales_General2021/ResultadosElectorales.csv",
#                  "./../datos/Solo_ResultadosProvisorios_bkp/ResultadosElectorales_PASO2021/ResultadosElectorales.csv",
#                          "./../datos/Solo_ResultadosProvisorios_bkp/ResultadosElectorales_General2017/ResultadosElectorales.csv",]

# for filepath in filepaths:
#     print(filepath)
#     with open(filepath, "r") as file:
#         df = pd.read_csv(file)
#     # print(df["cargo_nombre"].value_counts())
#     df["cargo_id"] = df["cargo_nombre"].map(mapping)
#     display(df["cargo_id"].fillna(-1).value_counts())
#     display(df["cargo_nombre"].fillna(-1).value_counts())
#     with open(filepath.replace("/Solo_ResultadosProvisorios_bkp/", "/Solo_ResultadosProvisorios/"), "w") as file:
#         df.to_csv(file, index=False)


In [3]:
csv_votos_pattern = '/home/matias/Documents/electoral/datos/Solo_ResultadosProvisorios/ResultadosElectorales_General2023/**/*.csv'
csv_votos_files = glob.glob(csv_votos_pattern, recursive=True)

In [4]:
# df_list = []
# for file in csv_votos_files:
#     df = pd.read_csv(file)
#     df_list.append(df)

# out = pd.concat(df_list, ignore_index=True)

In [5]:
# out.to_csv('/home/matias/Documents/electoral/datos/Solo_ResultadosProvisorios/ResultadosElectorales_General2023/ResultadosElectorales.csv')

In [6]:
# Define the glob pattern to find the CSV files
# csv_votos_pattern = './../datos/Solo_ResultadosProvisorios/**/*.csv'
csv_votos_pattern = '/home/matias/Documents/electoral/datos/Solo_ResultadosProvisorios/**/*.csv'

# Find all the CSV files that match the glob pattern
csv_votos_files = glob.glob(csv_votos_pattern, recursive=True)

In [7]:
csv_votos_files = [file for file in csv_votos_files if '2019' in file or '2023' in file or '2015' in file]
# csv_votos_files = [file for file in csv_votos_files if '2023' in file]

In [8]:
# Read all the CSV files into a list of DataFrames, lowercase their columns, and concatenate them
df_list = []
for file in csv_votos_files:
    df = pd.read_csv(file, dtype=object)
    df = df.rename(columns=lambda x: x.lower())
    df_list.append(df)

data = pd.concat(df_list, ignore_index=True)
# 7 minutes

In [9]:
# data.columns

# Index(['año', 'eleccion_tipo', 'recuento_tipo', 'padron_tipo', 'distrito_id',
#        'distrito_nombre', 'seccionprovincial_id', 'seccionprovincial_nombre',
#        'seccion_id', 'seccion_nombre', 'circuito_id', 'circuito_nombre',
#        'mesa_id', 'mesa_electores', 'mesa_tipo', 'cargo_id', 'cargo_nombre',
#        'agrupacion_id', 'agrupacion_nombre', 'lista_numero', 'lista_nombre',
#        'votos_tipo', 'votos_cantidad'],
#       dtype='object')

In [10]:
# data.to_csv('./full_data_bkp.csv', index=False)

# data = pd.read_csv('./full_data_bkp.csv') conviene correr el loop y generarlo.

In [13]:
    
def harmonize_agrupacion_id(agrupacion_id):
    if pd.isna(agrupacion_id):
        return agrupacion_id
    else:
        try:
            # First convert to float, then to int
            return str(int(float(agrupacion_id))).zfill(6)
        except ValueError:
            return agrupacion_id
    

In [15]:
### Clean the data and make necessary corrections to improve data quality (fillna, typecasting, etc.)

# Fill missing values in seccionprovincial_id with 0
data['seccionprovincial_id'].fillna(0, inplace=True)

# Convert distrito_id to int
data['distrito_id'] = data['distrito_id'].astype(int)

# Convention, fill with zeros to make circuit id 6 digits
data['circuito_id'] = data['circuito_id'].str.zfill(6)

# Apply the function to the 'agrupacion_id' column
data['agrupacion_id'] = data['agrupacion_id'].apply(harmonize_agrupacion_id)

# Convert seccion_id to int and change case of seccion_nombre to title case
data['seccion_id'] = data['seccion_id'].astype(int)
data['seccion_nombre'] = data['seccion_nombre'].fillna('')

In [16]:
data.head()

,año,eleccion_tipo,recuento_tipo,padron_tipo,distrito_id,distrito_nombre,seccionprovincial_id,seccionprovincial_nombre,seccion_id,seccion_nombre,...,mesa_tipo,cargo_id,cargo_nombre,agrupacion_id,agrupacion_nombre,lista_numero,lista_nombre,votos_tipo,votos_cantidad,unnamed: 0
0,2015,GENERAL,PROVISORIO,NORMAL,4,Córdoba,0,NaN,1,Capital,...,NATIVOS,2,SENADORES NACIONALES,000132,ALIANZA PROGRESISTAS,NaN,NaN,POSITIVO,13,NaN
1,2015,GENERAL,PROVISORIO,NORMAL,4,Córdoba,0,NaN,1,Capital,...,NATIVOS,2,SENADORES NACIONALES,000135,ALIANZA CAMBIEMOS,NaN,NaN,POSITIVO,170,NaN
2,2015,GENERAL,PROVISORIO,NORMAL,4,Córdoba,0,NaN,1,Capital,...,NATIVOS,2,SENADORES NACIONALES,000132,ALIANZA PROGRESISTAS,NaN,NaN,POSITIVO,10,NaN
3,2015,GENERAL,PROVISORIO,NORMAL,4,Córdoba,0,NaN,1,Capital,...,NATIVOS,2,SENADORES NACIONALES,000135,ALIANZA CAMBIEMOS,NaN,NaN,POSITIVO,171,NaN
4,2015,GENERAL,PROVISORIO,NORMAL,4,Córdoba,0,NaN,1,Capital,...,NATIVOS,2,SENADORES NACIONALES,000132,ALIANZA PROGRESISTAS,NaN,NaN,POSITIVO,16,NaN


In [17]:
# Generate an id for eleccion
gr_eleccion = ['año', 'eleccion_tipo', 'recuento_tipo', 'padron_tipo']
data['eleccion_id'] = data.groupby(gr_eleccion).ngroup()

# Casi 10 mins

In [18]:
# el_id_ = data[['año', 'eleccion_tipo', 'recuento_tipo', 'padron_tipo', 'eleccion_id']].drop_duplicates()

In [19]:
# els = pd.read_csv('./../datos/BD/eleccion_table.csv', dtype={'año': str})
# el_id_.merge(els, on = ['año', 'eleccion_tipo', 'recuento_tipo', 'padron_tipo'], how = 'left')[['eleccion_id_x']].T.to_dict()

In [20]:
# # dictionary

# map = {1: 5, 0: 4, 8: 17, 2: 6, 6: 14, 5: 13, 7: 18, 4: 12, 3: 11}


# data['eleccion_id'] = data['eleccion_id'].map(map)
# data['eleccion_id'] = 18

In [21]:
## Harmonize 'votos_tipo' column
import json

with open('./../datos/correccion/valor_votos_tipo.json', 'r') as f:
    valor_votos_tipo_homog = json.load(f)

data['votos_tipo'] = data['votos_tipo'].map(valor_votos_tipo_homog)

In [22]:
bd_dir = './../datos/BD151923'

## Decomposition into relational tables

In [23]:
def decompose_data(data, group_cols, table_cols, table_name, bd_dir = './../datos/BD151923'):
    """
    Creates a table with unique values from the specified columns of the given data, grouped by the specified columns.
    Saves the resulting table as a CSV with the specified name.
    Removes the specified columns from the original data.
    
    Args:
        data (pd.DataFrame): Input DataFrame
        group_cols (list): List of columns to group by
        table_cols (list): List of columns to include in resulting table
        table_name (str): Name of table to save
    
    Returns:
        None
    """

    # Create table with unique values
    table_df = data[group_cols + table_cols].drop_duplicates().reset_index(drop=True)
    
    # Save table as CSV
    table_df.to_csv(bd_dir + '/' + table_name + "_table.csv", index=False)
    
    # Remove table_cols from original data
    data.drop(table_cols, axis=1, inplace=True)


In [24]:
# data_bkp = data.copy() # In case we make a mistake or wish to change something in the following cells without running from the start

In [25]:
# Create a table for eleccion
decompose_data(data, ['eleccion_id'], ['año', 'eleccion_tipo', 'recuento_tipo', 'padron_tipo'], 'eleccion')

In [26]:

# Create a table for seccion
decompose_data(data, ['distrito_id', 'seccion_id', 'seccionprovincial_id'], ['seccion_nombre'], 'seccion')


In [27]:

# Create a table for seccionprovincial
decompose_data(data, ['distrito_id', 'seccionprovincial_id'], ['seccionprovincial_nombre'], 'seccionprovincial')

In [28]:

# Create a table for distrito
decompose_data(data, ['distrito_id'], ['distrito_nombre'], 'distrito')

In [29]:

# Create a table for mesas
decompose_data(data, ['eleccion_id', 'distrito_id', 'seccion_id', 'circuito_id', 'mesa_id'], ['mesa_electores', 'mesa_tipo'], 'mesas')

In [30]:

# Create a table for cargo
decompose_data(data, ['cargo_id'], ['cargo_nombre'], 'cargo')

In [31]:

# Create a table for agrupacion
decompose_data(data, ['eleccion_id', 'distrito_id', 'cargo_id', 'agrupacion_id', 'votos_tipo', 'lista_numero'], ['lista_nombre'], 'agrupacion_lista')  # lista nombre no deberia quedar porque puede ser mismo numero con distinto lista nombre

In [32]:

# Create a table for agrupacion
decompose_data(data, ['eleccion_id', 'distrito_id', 'cargo_id', 'agrupacion_id', 'votos_tipo'], ['agrupacion_nombre'], 'agrupacion_nombre')  # lista nombre no deberia quedar porque puede ser mismo numero con distinto lista nombre

In [33]:
## 
# Cargo, revisar que el 7 deberia ser intendente y sale concejal

In [34]:
# Create a table for circuitos. # Se admite que cambien de eleccion a eleccion
decompose_data(data, ['eleccion_id', 'distrito_id', 'seccion_id', 'seccionprovincial_id', 'circuito_id'], ['circuito_nombre'], 'circuito')

In [35]:
data = data.drop(['unnamed: 0'], axis = 1)

In [36]:
data.shape

(45881361, 11)

In [37]:
data.head()

,distrito_id,seccionprovincial_id,seccion_id,circuito_id,mesa_id,cargo_id,agrupacion_id,lista_numero,votos_tipo,votos_cantidad,eleccion_id
0,4,0,1,00011D,1807,2,000132,NaN,POSITIVO,13,1
1,4,0,1,00011D,1807,2,000135,NaN,POSITIVO,170,1
2,4,0,1,00011D,1808,2,000132,NaN,POSITIVO,10,1
3,4,0,1,00011D,1808,2,000135,NaN,POSITIVO,171,1
4,4,0,1,00011D,1814,2,000132,NaN,POSITIVO,16,1


In [ ]:
xx

NameError: name 'xx' is not defined

## Harmonize names of Cargo, Distrito, Seccion.

In [38]:
def names_harmonization(df, id_cols, name_col):
    df[name_col] = df[name_col].str.title() ###
    df[name_col] = df.dropna().groupby(id_cols)[name_col].transform(lambda x: x.mode()[0])
    df.drop_duplicates(subset=id_cols, inplace=True)
    return df

# Load the seccion table and apply name harmonization
seccion_df = pd.read_csv(bd_dir + '/seccion_table.csv'); print(seccion_df.shape)
seccion_df = names_harmonization(seccion_df, ['distrito_id', 'seccion_id'], 'seccion_nombre')
seccion_df.to_csv(bd_dir + '/seccion_table.csv', index=False)

# Load the cargo table and apply name harmonization
cargo_df = pd.read_csv(bd_dir + '/cargo_table.csv'); print(cargo_df.shape)
cargo_df = names_harmonization(cargo_df, ['cargo_id'], 'cargo_nombre')
cargo_df.to_csv(bd_dir + '/cargo_table.csv', index=False)

# Load the distrito table and apply name harmonization
distrito_df = pd.read_csv(bd_dir + '/distrito_table.csv'); print(distrito_df.shape)
distrito_df = names_harmonization(distrito_df, ['distrito_id'], 'distrito_nombre')
distrito_df.to_csv(bd_dir + '/distrito_table.csv', index=False)

# Load the mesas table and apply name harmonization
df_mesas = pd.read_csv(bd_dir + '/mesas_table.csv')
df_mesas['mesa_tipo'] = df_mesas['mesa_tipo'].replace('NATIVO', 'NATIVOS').replace('EXTRANJERO', 'EXTRANJEROS')
df_mesas = df_mesas.drop_duplicates()
df_mesas.to_csv(bd_dir + '/mesas_table.csv', index=False)


(1096, 4)
(24, 2)
(27, 2)


## Save Votos datasets

In [39]:
for eleccion_id, df in data.groupby('eleccion_id'):
    filename = f'{bd_dir}/votos_eleccion_{eleccion_id}_table.csv'
    df = df.drop_duplicates() # Especially because of 2021
    print(df.shape)
    df.to_csv(filename, index=False)

(565020, 11)
(5053072, 11)
(8956666, 11)
(264145, 11)
(3159297, 11)
(303176, 11)
(5082019, 11)
(5868102, 11)
(16629864, 11)


In [ ]:
import pandas as pd


In [ ]:
filename = './../datos/BD151923/votos_eleccion_7_table.csv'
info7 = pd.read_csv(filename, usecols = ['eleccion_id', 'distrito_id', 'agrupacion_id', 'votos_tipo', 'cargo_id', 'votos_cantidad'])

In [ ]:
filename = './../datos/BD151923/votos_eleccion_8_table.csv'
info8 = pd.read_csv(filename, usecols = ['eleccion_id', 'distrito_id', 'agrupacion_id', 'votos_tipo', 'cargo_id', 'votos_cantidad'])

In [ ]:
info7.groupby(['cargo_id']).apply(lambda x: x.agrupacion_id.astype(str).str.len().value_counts())
#GRAL

cargo_id  agrupacion_id
1         3                522600
          1                522600
2         1                246245
          5                156748
          6                 35386
3         1                522600
          5                267792
          6                168630
4         1                206295
          5                163978
5         1                102530
          5                 81255
6         1                107535
          5                 83323
7         1                210680
          5                163766
          6                 13964
8         3                522600
          1                522600
9         1                522600
          5                266734
          6                168119
10        6                 39288
          1                  9435
          5                  3365
11        1                 43850
          5                 35080
12        1                 43850
          5             

In [ ]:
# maximum total votes per cargo_id and agrupacion_id
info7.groupby(['cargo_id', 'agrupacion_id'])['votos_cantidad'].sum().sort_values(ascending = False).head(10)



cargo_id  agrupacion_id
1         134              9645983
8         134              9185701
1         135              7884336
8         135              7368506
          132              6273761
1         132              6267152
4         20134            4233092
7         20134            4159180
2         20134            4032180
9         20134            3996899
Name: votos_cantidad, dtype: int64

In [ ]:
info8.groupby(['cargo_id']).apply(lambda x: x.agrupacion_id.astype(str).str.len().value_counts())
#PASO

cargo_id  agrupacion_id
1         3                1450515
          2                1141502
          1                 522660
2         3                1186404
          2                 317974
          1                 247200
3         3                1505757
          2                 692055
          1                 530941
4         3                 860367
          1                 210850
          2                 162773
5         3                 436835
          1                 107085
          2                  75163
6         3                 432194
          1                 112090
          2                  82118
7         3                 652917
          1                 206185
          2                 123639
          4                   1690
8         3                1098026
          2                1038640
          1                 522660
9         3                1722641
          2                 641937
          1                 530

In [ ]:
# maximum total votes per cargo_id and agrupacion_id
info8.groupby(['cargo_id', 'agrupacion_id'])['votos_cantidad'].sum().sort_values(ascending = False).head(10)



cargo_id  agrupacion_id
1         135              7116352
          132              6698029
8         135              6596288
          132              6562071
1         134              6460689
8         134              6146848
3         132              4723349
9         132              4676061
3         134              3436391
9         134              3432037
Name: votos_cantidad, dtype: int64

In [ ]:
count_check = data.groupby('eleccion_id').count()

In [ ]:
count_check.div(count_check.max(1), 0) ## Los faltantes en agrupacion id y lista id tienen origen en la fuente de datos
## Los numeros de lista son para elecciones primarias, no para elecciones generales.
## Ademas, no hay lista o agrupacion para votos no positivos (blancos, nulos, recurridos, etc.)

,distrito_id,seccionprovincial_id,seccion_id,circuito_id,mesa_id,cargo_id,agrupacion_id,lista_numero,votos_tipo,votos_cantidad
eleccion_id,,,,,,,,,,
0,1.0,1.0,1.0,1.0,1.0,1.0,0.640411,0.001853,1.0,1.0
1,1.0,1.0,1.0,1.0,1.0,1.0,0.736808,0.119236,1.0,1.0
2,1.0,1.0,1.0,1.0,1.0,1.0,0.606258,0.000000,1.0,1.0
3,1.0,1.0,1.0,1.0,1.0,1.0,0.677434,0.677434,1.0,1.0
4,1.0,1.0,1.0,1.0,1.0,1.0,0.333333,0.000000,1.0,1.0
5,1.0,1.0,1.0,1.0,1.0,1.0,0.574375,0.003887,1.0,1.0
6,1.0,1.0,1.0,1.0,1.0,1.0,0.763525,0.726825,1.0,1.0
7,1.0,1.0,1.0,1.0,1.0,1.0,0.000000,0.000000,1.0,1.0
8,1.0,1.0,1.0,1.0,1.0,1.0,0.610419,0.000000,1.0,1.0


In [ ]:
# for file in csv_votos_files[:4]:
#     print(file)
#     df = pd.read_csv(file, usecols = ['año', 'eleccion_tipo', 'recuento_tipo', 'padron_tipo', 'agrupacion_id', 'lista_numero', 'agrupacion_nombre', 'lista_nombre'])
#     print(df.dtypes)
#     print(df.count())
#     # Create table with unique values
#     table_df = df[['año', 'eleccion_tipo', 'recuento_tipo', 'padron_tipo', 'agrupacion_id', 'lista_numero'] + ['agrupacion_nombre', 'lista_nombre']].drop_duplicates().reset_index(drop=True)
    
#     print(table_df.dtypes)

#     display(pd.Series(df.agrupacion_id.unique()).sample(50).values)
#     display(pd.Series(table_df.agrupacion_id.unique()).sample(50).values)
#     print('########################################')